In [1]:
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes


In [2]:
import json
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import torch

In [3]:
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH = "/content/astrology_chats_fixed.jsonl"
OUTPUT_DIR = "qwen-astrologer-lora"
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 3
LR = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 16

In [4]:
##1. Helper: convert ShareGPT-style export to the format this script wants
def convert_sharegpt(in_path: str, out_path: str, system_prompt: str):


  """Run this once if your drive data is in ShareGPT format:
  [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]
  """
  role_map = {"human": "user", "gpt": "assistant", "system": "system"}
  with open(in_path, "r", encoding="utf-8") as f_in, \
        open(out_path, "w", encoding="utf-8") as f_out:
          for line in f_in:
            row = json.loads(line)
            convo = row.get("conversations", row.get("conversation", []))
            msgs = [{"role": "system", "content": system_prompt}]
            for turn in convo:
                role = role_map.get(turn.get("from"), turn.get("from"))
                msgs.append({"role": role, "content": turn.get("value", "")})
            f_out.write(json.dumps({"conversations": msgs}, ensure_ascii=False) + "\n")


In [5]:
SYSTEM_PROMPT = (
    "You are an expert Vedic astrologer. You know how to read a birth chart "
    "(kundli), you speak warmly and empathetically, and when asked for a "
    "prediction you first ask for accurate birth details (date, time, place) "
    "before analysing the kundli."
)

In [6]:
##LOADING TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token


def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

raw_ds = load_dataset("json", data_files=DATA_PATH, split="train")
ds = raw_ds.map(format_example, remove_columns=raw_ds.column_names)



Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

In [7]:
# Simple train/eval split
ds = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = ds["train"], ds["test"]

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [9]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,No log,1.825499,1.778916,27904.000000,0.600913
2,No log,1.722372,1.903462,55808.000000,0.616889
3,1.599498,1.668871,1.817125,83712.000000,0.629605


TrainOutput(global_step=12, training_loss=1.5155484875043232, metrics={'train_runtime': 2370.2124, 'train_samples_per_second': 0.062, 'train_steps_per_second': 0.005, 'total_flos': 3571650823716864.0, 'train_loss': 1.5155484875043232, 'epoch': 3.0})

In [10]:
trainer.save_model(OUTPUT_DIR)          # saves LoRA adapter
tokenizer.save_pretrained(OUTPUT_DIR)

# Merge LoRA into base weights so vLLM can load a normal HF model dir
merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}-merged", safe_serialization=True)
tokenizer.save_pretrained(f"{OUTPUT_DIR}-merged")

print("Done. Merged model ready at:", f"{OUTPUT_DIR}-merged")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done. Merged model ready at: qwen-astrologer-lora-merged


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
!ls "/content/drive/MyDrive/Colab Notebooks/"

'Copy of FINE TUNING QWEN.ipynb'   ML_Lab_File_Complete.ipynb
'FINE TUNING QWEN.ipynb'	   Untitled0.ipynb


In [17]:
import json

path = "/content/drive/MyDrive/Colab Notebooks/FINE TUNING QWEN.ipynb.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

metadata = nb.get("metadata", {})
if "widgets" in metadata:
    del metadata["widgets"]
    print("Removed widgets metadata")
nb["metadata"] = metadata

with open(path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print("Fixed notebook saved")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/FINE TUNING QWEN.ipynb.ipynb'